<img src="./logo_UNSAM.jpg" align="right" width="150" /> 

#### Análisis y Procesamiento de Señales
# Trabajo Final
# “Elección de un filtro para el análisis de señales de EEG”
#### Tomás Altimare Bercovich

# Cargado de Datos Previo

In [3]:
# %% --- Bibliotecas ---
import os # Se usa en EDF_to_RAW
import mne # https://mne.tools/stable/index.html
from mne_connectivity import spectral_connectivity_epochs
from mne.preprocessing import ICA # Para eliminar los artefactos (principalmente ruidos por pestañeo)
import seaborn as sns # https://seaborn.pydata.org/generated/seaborn.heatmap.html
import matplotlib.pyplot as plt
import scipy.signal as sig
import scipy.stats as stats
import numpy as np
from pytc2.sistemas_lineales import plot_plantilla # Plantilla de diseño para filtros FIR 
import networkx as nx

# %% --- Variables Globales ---
fs = 250
canales = np.array(['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T3', 'C3', 'Cz', 'C4', 'T4', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2'])

# %% --- Conversión de archivos EDF a RAW ---
"""
h01_edf, h02, ... = sujetos saludables
s01, s02, ... = sujetos con esquizofrenia paranoide
"""
sujetos_sanos = {}
sujetos_con_esquizofrenia = {}
sujetos_h = np.array(['h01','h02','h03','h04','h05','h06','h07','h08','h09','h10','h11','h12','h13','h14'])
sujetos_s = np.array(['s01','s02','s03','s04','s05','s06','s07','s08','s09','s10','s11','s12','s13','s14'])

for i in range(1, 15):
    # El f"{i:02d}" le da el formato de 01, 02... 10, 11
    id_sano = f"h{i:02d}" 
    ruta_archivo_sano = f"Sujetos/{id_sano}.edf"
    id_esquizofrenia = f"s{i:02d}" 
    ruta_archivo_esquizofrenia = f"Sujetos/{id_esquizofrenia}.edf"
    
    sujetos_sanos[id_sano] = EDF_to_RAW(ruta_archivo_sano, sujeto=id_sano)
    sujetos_con_esquizofrenia[id_esquizofrenia] = EDF_to_RAW(ruta_archivo_esquizofrenia, sujeto=id_esquizofrenia)

# print(f"sujetos cargados correctamente: \n {list(sujetos_sanos.keys())} \n {list(sujetos_con_esquizofrenia.keys())}")
print("sujetos cargados correctamente")

ModuleNotFoundError: No module named 'networkx'

# Introducción Teórica
El electroencefalograma o “EEG” es un estudio no invasivo y seguro que mide la actividad eléctrica cerebral utilizando electrodos distribuidos sobre la cabeza del paciente. En este trabajo se buscará replicar el preprocesamiento realizado en ‘Graph-based analysis of brain connectivity in schizophrenia’ por Elzbieta Olejarczyk y Wojciech Jernajczyk[1] en el cual se realizaron 28 EEG: 14 pacientes con esquizofrenia paranoide (EP) y 14 pacientes sanos de control, con el objetivo de comparar la actividad eléctrica cerebral entre ambos grupos. Se buscará analizar el filtrado realizado por los autores, proponiendo una alternativa con el objetivo de observar si las diferencias en sus características muestran resultados distintos al analizar y comparar las señales.

La señal de EEG tiene la característica de ser una señal de amplitud relativamente baja (del orden de 10-5, como puede observarse en la Figura 1.1), y por ende, con alta variabilidad y un alto componente de ruido. La información relevante de esta señal se encuentra en el rango de frecuencias entre 2 y 45 Hz, separándose usualmente en las siguientes bandas : 2–4 Hz (delta), 4.5–7.5 Hz (theta), 8–12.5 Hz (alpha), 13–30 Hz (beta), 30–45 Hz (gamma).


In [ ]:
# %% --- Visualización de la señal en dominio temporal ---
fig = sujetos_sanos['h01'].plot(duration=5, n_channels=19, scalings='auto', title='EEG Esquizofrenia')
plt.show()

ts_signal = sujetos_sanos['h01'].get_data()[0]
plt.figure()
plt.title('Señal EEG en el dominio Temporal', fontsize = 20)
plt.xlabel('Tiempo [s]', fontsize = 18)
plt.ylabel('Amplitud', fontsize = 18)
plt.plot(ts_signal, color = '#363636', label = 'Señal EEG: Sujeto H01 - Canal FP1')
plt.grid(True, which='both', ls=':')
plt.tick_params(axis='both', labelsize=16)
plt.xlim([75700, 76300])
plt.ylim([-2*10**(-5), 2*10**(-5)])
plt.legend(fontsize = 14)
plt.show()

_Figura 1.1. Fragmento de señal EEG sin filtrar. Proveniente del electrodo colocado en el lóbulo frontal FP1 del sujeto de control H01._

Al capturar una señal EEG, el parpadeo ocular tiende a ser uno de los artefactos más comunes[10], debido a la posición de los electrodos. Por ejemplo, los electrodos FP1 y FP2, los cuales se encuentran muy cercanos a la frente. Con el objetivo de eliminar estos artefactos de ruido, se utiliza un método computacional para detectar componentes independientes en la señal (ICA).

Una vez obtenidas las señales sin artefactos, las bandas de frecuencia de interés son separadas mediante filtros pasa banda. El diseño de estos filtros debe contemplar las características básicas de un filtro (ver Figura 1.2).

<img src="./caracteristicas_filtros_holton.jpg" align="middle" width="800" /> 
_Figura 1.2. Características de un Filtro [8]_

Al momento de diseñar un filtro se deben tener en cuenta 4 factores principales a definir: la frecuencia de corte del mismo (en este caso son dos) según la banda de frecuencias que se quiera dejar pasar (banda de paso), el ancho de la banda de transición, la atenuación de la banda de ‘stop’ y el ripple en la banda de paso.

Una vez diseñados los filtros, es importante corroborar su correcto diseño mediante el análisis de su respuesta en frecuencia de su módulo, fase y retardo de grupo. A su vez, se debe corroborar su funcionamiento mediante análisis espectrales de potencia (PSD).


# Datos
El estudio fue realizado en pacientes con un rango etario de 23 a 33 años, siendo la  mitad hombres (mitad sanos, y mitad con EP) y la otra mitad mujeres (mitad sanas, y mitad con EP). Para cada paciente, el estudio mantuvo una duración de 15 minutos con los ojos cerrados y en posición de descanso, buscando minimizar el ruido por movimientos corporales o pestañeo. Por otro lado, las señales fueron adquiridas mediante una frecuencia de muestreo de 250 Hz, utilizando un montaje estándar 10-20 de EEG con 19 canales: Fp1, Fp2, F7, F3, Fz, F4, F8, T3, C3, Cz, C4, T4, T5, P3, Pz, P4, T6, O1, O2. El electrodo de referencia fue ubicado en FCz (es decir, entre medio de los electrodos “Fz” y “Cz”), como puede observarse en la figura 2.

<img src="./eeg_esquema.jpg" align="middle" width="800" /> 
_Figura 2.1. Esquema de montaje estándar 10-20 de EEG utilizado_

El análisis de las señales fue realizado a partir de segmentos seleccionados por los autores. Ésto con el propósito de analizar las señales de EEG con la menor cantidad de artefactos posible (como movimiento de ojos, actividad cardíaca o contracciones musculares). A partir de estos segmentos de señal, se aplicó un filtro Butterworth de segundo orden en las siguientes bandas de frecuencia: 2–4 Hz (delta), 4.5–7.5 Hz (theta), 8–12.5 Hz (alpha), 13–30 Hz (beta), 30–45 Hz (gamma).

# Metodología
### Estructura del código
El procesamiento de las señales fue realizado en Spyder 6 mediante Python, utilizando las siguientes bibliotecas:
- matplotlib: Realización de gráficos y figuras.
- scipy: Utilizada para aplicar ventanas (sig.windows), conseguir la respuesta en frecuencia de filtros (sig.freqz), diseño (sig.iirdesign) y aplicación (sig.filtfilt) filtros IIR y diseño de filtros Butterworth (sig.butter).
- numpy: Utilizada para diversas funciones matemáticas (np.abs, np.diff, …), estadísticas (np.mean), y para generar estructuras de datos como un array.
- mne: Utilizada para procesar y visualizar los EEG, eliminar artefactos y calcular correlaciones (como el PLI).
- seaborn: Utilizado para generar las matrices de conectividad.

La estructura del código está dada por los siguientes dos archivos:
- main.py: Programa principal, el cual contiene el diseño y la aplicación de los filtros, la eliminación de artefactos y la el armado de las matrices de conectividad.
- funciones.py: Archivo donde se alojan todas las funciones y bibliotecas utilizadas en el transcurso del programa principal, “main.py”.

### Preparado de los Datos
Al igual que en el estudio[1], se comenzó realizando una conversión en el formato de los archivos, pasando de archivos tipo “EDF” a “RAW”. Para ello, se implementó la función EDF_to_RAW, generada a partir de la herramienta mne.io.read_raw_edf de la biblioteca MNE.

### Eliminación de Artefactos
Para la eliminación de artefactos del EEG (principalmente artefactos debidos al movimiento ocular) se utilizó la herramienta ICA (Independent Component Analysis) de la biblioteca MNE preprocessing. La herramienta es sensible a las bajas frecuencias, por ello, la página de MNE[7] recomienda aplicar un filtro pasa altos con una frecuencia de corte de 1 Hz sobre la señal antes de utilizar la función. Para filtrado se decidió utilizar un filtro IIR Butterworth debido a que las exigencias sobre su diseño son bajas, por lo que se ponderó el rápido diseño y bajo costo computacional del mismo. Su funcionamiento fue constatado mediante un análisis en frecuencia del filtro (ver Figura 3.1). Debido a que la señal es filtrada de forma ‘offline’, se realizó un filtrado bidireccional mediante ‘sosfiltfilt’. Este método se basa en aplicar el filtro “hacia adelante” en el tiempo y luego “hacia atrás”. Tiene la ventaja de garantizar una respuesta de fase nula, y por ende, no induce un retardo en la señal en el dominio del tiempo. Por otro lado, es importante destacar para su diseño que, al estar aplicando el filtro dos veces, la atenuación del mismo en la banda de ‘stop’ y el ripple en su banda de paso se verán duplicados. Una vez filtrada la señal, se detectaron y eliminaron los artefactos mediante la herramienta mencionada para cada una de las señales.

In [ ]:
# %% --- Elimino artefactos con ICA ---
def limpiar_artefactos_ica(lista_sujetos, dict_crudo, fs):
    """
    Descripción: Aplica ICA a las señales continuas para detectar y eliminar 
    automáticamente los artefactos de parpadeo usando Fp1 como referencia.
    """
    dict_limpio = {}
    print("\nIniciando extracción de artefactos con ICA...")
    
    cant_coef_hp = 5601
    
    ripple = 0.1   # dB
    attenuation = 25  # dB
    
    fstop_hp = 0.7  # Hz
    fpass_hp = 1  # Hz
    
    frecs_hp = [0.0, fstop_hp, fpass_hp, fs/2]
    gains_db_hp = [-np.inf, -attenuation, -ripple, 0]
    gains_hp = 10**(np.array(gains_db_hp)/20)
    
    num_hp = sig.firls(numtaps = cant_coef_hp, bands = frecs_hp, desired = gains_hp, fs = fs)
    
    sos = sig.iirdesign(wp = 1.1, ws = 0.7, gpass = 0.5, gstop = 30, 
                        analog = False, ftype = 'butter', output = 'sos', fs = fs)

    f, h = sig.sosfreqz(sos, worN=4000, fs=fs)

    magnitud_db = 20 * np.log10(np.abs(h) + 1e-15)
    
    # --- Configuración del Ploteo ---
    plt.figure(figsize=(10, 6))    
    plt.plot(f, magnitud_db, label='Filtro pasa altos')
    plt.axhline(-3, color='red', ls='--', alpha=0.5, label='Atenuación de la frecuencia de corte (ω0)')
    plt.axhline(-30, color='red', ls='--', alpha=0.5, label='Atenuación mínima (gstop = -30 dB)')
    plt.title('Respuesta en Magnitud', fontsize=20)
    plt.xlabel('Frecuencia [Hz]', fontsize=18)
    plt.ylabel('|H(f)| [dB]', fontsize=18)
    plt.xlim([0, 2]) 
    plt.ylim([-40, 1])
    ticks_base = list(np.arange(-40, 1, 10))
    nuevos_ticks_y = sorted(ticks_base + [-3.0]) 
    plt.yticks(nuevos_ticks_y) 
    plt.tick_params(axis='both', labelsize=14)
    plt.grid(True, which='both', ls=':')
    plt.legend(fontsize=14)
    plt.show()
    
    for sujeto in lista_sujetos:
        raw_sujeto = dict_crudo[sujeto].copy()
         
        matriz_cruda = raw_sujeto.get_data()
        
        matriz_filtrada = sig.sosfiltfilt(sos, matriz_cruda, axis=1) #1 porque es un filt
        
        #matriz_filtrada = sig.filtfilt(b, a, matriz_cruda, axis=1)
        
        info_actualizada = raw_sujeto.info.copy()
        
        try:
            info_actualizada['highpass'] = 1
        except RuntimeError:
            with info_actualizada._unlock():
                info_actualizada['highpass'] = 1
        
        raw_para_ica = mne.io.RawArray(matriz_filtrada, info_actualizada, verbose=False)
        
        # 3. Configuramos el motor matemático ICA
        # Le pedimos que encuentre 15 componentes independientes (voces)
        ica = ICA(n_components=15, method='fastica', random_state=42, max_iter='auto')
        ica.fit(raw_para_ica, verbose=False)
        
        # 4. Búsqueda automática del parpadeo (Usamos Fp1 como "buchón" o EOG virtual)
        indices_malos, scores = ica.find_bads_eog(raw_sujeto, ch_name='Fp1', verbose=False)
        
        # Le decimos a ICA cuáles son los componentes que hay que borrar
        ica.exclude = indices_malos
        
        # 5. Aplicamos la limpieza a nuestra señal original
        raw_limpio = raw_sujeto.copy()
        ica.apply(raw_limpio, verbose=False)
        
        # Guardamos el raw limpio en nuestro nuevo diccionario
        dict_limpio[sujeto] = raw_limpio
        print(f"[{sujeto}] ICA completado: Se borraron {len(indices_malos)} componentes de parpadeo.")
        
    return dict_limpio

sujetos_sanos_ICA = limpiar_artefactos_ica(sujetos_h, sujetos_sanos, fs=fs)
sujetos_esquizofrenia_ICA = limpiar_artefactos_ica(sujetos_s, sujetos_con_esquizofrenia, fs=fs)

# CAR (Common Average Reference):
for sujeto in sujetos_h:
    # Esto resta el promedio de los 19 canales a cada canal individual
    sujetos_sanos_ICA[sujeto].set_eeg_reference('average', projection=False, verbose=False)

for sujeto in sujetos_s:
    sujetos_esquizofrenia_ICA[sujeto].set_eeg_reference('average', projection=False, verbose=False)

### Filtrado
Con el objetivo de replicar el filtrado realizado en el estudio[1], se comenzó diseñando un filtro IIR Butterworth de segundo orden mediante el método ‘sig.butter’ de scipy signal para cada banda de frecuencias de las mencionadas anteriormente: ‘Delta’, ‘Theta’, ‘Alpha’, ‘Beta’ y ‘Gamma’. El mismo fue elegido sobre iirdesign debido a que permite crear el filtro Butterworth a partir de su orden, y no según la atenuación que efectúa o la banda de transición del mismo. Una vez diseñados los filtros, se analizó su respuesta en frecuencia (ver Figura 3.2), encontrando lo esperado: filtros con una caída muy suave en su banda de transición.

Para analizar si la velocidad de caída en la banda de transición del filtro supone una superposición de las bandas de interés (es decir, que no logren separarse correctamente), se diseñó un filtro IIR con caída más abrupta: el Chebyshev tipo 2. Una vez realizado el diseño de los nuevos filtros, se visualizó y comparó su respuesta en frecuencia con el filtro Butterworth de segundo orden (ver Figura 3.3).

Dado que el filtrado es realizado luego de haber obtenido la señal, el uso del método de filtrado bidireccional resulta apropiado. En un sistema en el que se requiriera filtrar la señal a medida que se obtiene, este método no podría ser utilizado debido que requiere utilizar una señal finita, por lo que utilizar un filtro IIR introduciría un retardo variable en la misma. En este caso, lo recomendable sería utilizar un filtro FIR (Finite Impulse Response), cuya salida depende únicamente de valores presentes en la entrada, sin realimentación. Además, su fase es lineal por construcción, por lo que mantendría un retardo constante, aunque a costa de un orden mucho mayor (y por ende, un mayor costo computacional).

Una vez diseñados los filtros, fueron aplicados a la señal mediante un filtrado bidireccional (hacia ‘adelante’ y hacia ‘atrás’) utilizando la función ‘filtfilt’ de scipy signal. Este método de filtrado garantiza una fase perfectamente lineal, sin desfase temporal, corrigiendo la principal desventaja de los filtros IIR. 

Los filtros fueron aplicados por separado en un único canal para las distintas bandas de frecuencias de un único sujeto. Su funcionamiento fue verificado mediante un análisis espectral de potencias, comparado el espectro de la señal antes y luego de aplicar los diferentes filtros, viendo que los mismos actúen según lo diseñado y comparando los resultados entre los dos tipos de filtro para una misma banda (ver Figura 3.4). Para realizar la estimación espectral de las señales se optó por utilizar el método de Welch. Se eligió sobre el método de periodograma modificado debido a que el análisis se centró en la señal en un grupo o banda de frecuencias, por lo que no se requirió analizar en detalle la energía de la señal a una frecuencia en concreto. A su vez, el método de Welch tiene la ventaja de utilizar promediado y solapamiento, obteniendo una menor varianza con respecto al método del periodograma modificado.

Tras verificar que el funcionamiento de los filtros fuera el deseado, se procedió a aplicarlos en los 19 canales para los 28 sujetos (532 señales), terminado el pre-procesado de las señales para poder pasar a ser analizadas y comparadas.

### Correlación de Señales y Análisis de Redes
Con el objetivo de realizar un análisis comparativo entre las poblaciones de sujetos de control y con esquizofrenia, se buscó representar el comportamiento de cada cerebro como una red compleja o grafo, interpretando cada uno de los electrodos como un ‘nodo’ y definiendo su correlación y causalidad mediante ‘enlaces’. El análisis de estas redes determina a grandes rasgos la conectividad entre las distintas regiones del cerebro para cada uno de los sujetos, por lo que el objetivo de este análisis será analizar si existe una diferencia de conectividad entre las poblaciones. A su vez, debido a que el enfoque de la materia está destinado al preprocesamiento de las señales, se decidió realizar una segunda comparativa en simultáneo, comparando las señales filtradas mediante los filtros Butterworth y los filtros Chebyshev, buscando ver si las características del filtrado implican una diferencia al correlacionar las señales. 

Para definir la correlación y causalidad entre los nodos (canales del EEG), se replicó la metodología utilizada en el estudio[1] implementando una herramienta que mide la correlación temporal entre dos señales teniendo en cuenta la sincronización de sus fases: Phase-Lag Index (PLI). Éste método tiene el objetivo de tener en cuenta el comportamiento biológico de la señal: siendo que los impulsos nerviosos no son instantáneos, por lo que si dos electrodos lejanos detectaran actividad en un mismo instante, éste método se encarga de descartar su causalidad, debido a que, si bien son dos eventos que fueron correlacionados en un instante, es improbable que ambos electrodos hayan detectado la misma actividad (debido a su lejanía). Teniendo este factor en cuenta, el PLI cuantifica la sincronización de fase entre dos señales, devolviendo un valor entre 0 (para 2 eventos no correlacionados) y 1 (para 2 eventos perfectamente correlacionados). Se define como:

$$ PLI = \left| \frac{1}{N} \sum_{n=1}^{N} \text{sign}(\Delta\phi(t_n)) \right| $$

Siendo (tn)=x(tn)-y(tn) la diferencia de fase entre dos señales x(t) e y(t) en un instante tn, pudiendo tomar valores entre -π<∆Φ(tn)<π, y donde n=1, ..., N es la cantidad de muestras en el dominio del tiempo.

Para generar las matrices mediante este método, se debió dividir las señales en épocas. La duración de cada época es definida en el estudio con una duración de 30 segundos, pero al avanzar en la metodología se notó que este lapso de tiempo elegido devolvía una cantidad muy baja de épocas para la duración en segundos de las señales (siendo que la señal EEG con menor duración es de 740 segundos). Esta baja cantidad de épocas se notó al aplicar el análisis de redes, a partir del cual se definió utilizar épocas de 5 segundos. El valor de 5 segundos fue definido criteriosamente teniendo en cuenta que: con 5 segundos, es posible captar al menos 10 periodos de las señales de más baja frecuencia (las Delta, que van de 2-4 Hz), y a su vez, para una señal de 740 segundos (la más corta en la base de datos), se logran obtener una cantidad de aproximadamente 150 épocas, lo cual resulta una cantidad mucho mayor a las aproximadamente 25 que se pudieran obtener con épocas de 30 segundos. Una vez fueron traducidas a épocas, a partir de las señales y utilizando el método de PLI se generó para cada uno de los sujetos una matriz de conectividad conformada con las correlaciones de la señal obtenida por cada uno de los 19 canales, obteniendo matrices de 19x19 (ver Figura 3.5). 

Tras obtener las 56 matrices de correlación (dos por sujeto, siendo una por cada filtro) para una misma banda de frecuencias, se procedió a modelar cada una como un grafo. Para definir los enlaces del mismo, se debe determinar un valor de umbral que defina la conexión entre dos nodos como ‘conectados’. Este umbral fue definido, al igual que en el estudio, realizando un análisis de datos subrogados con un valor de significancia del 0.05, descartando a cero todas las conexiones que no cumplieran con él. El método de análisis de datos subrogados se basa en mantener la amplitud de la señal, y al mismo tiempo, eliminar la relación de fase entre las señales originales mediante la reordenación aleatoria de los datos en el dominio de la frecuencia utilizando la transformada rápida de fourier (FFT). Posteriormente, se vuelve al dominio del tiempo mediante la transformada inversa (iFFT). La significancia fue testeada a partir de comparar los valores subrogados obtenidos con los originales (filtrados). El número de subrogados que fue calculado para cada señal fue de 100.

Una vez obtenidas las matrices de correlación correctamente binarizadas, se representaron mediante redes complejas conformadas por nodos y enlaces (ver Figura 3.6), realizando un análisis de las características de cada red con el objetivo de compararlas y ver si hay diferencias en la cantidad de enlaces que se puedan considerar significativos utilizando diferentes filtrados en la señal.



# Resultados
Para el diseño del filtro pasa altos que sería utilizado antes de realizar la eliminación de artefactos se definió un ripple en la banda de paso de 0.5 (gpass = 0.5) y una atenuación en la banda de ‘stop’ de 30 dB (gstop = 30). Definiendo la frecuencia de corte (1 Hz) como la frecuencia para la cual la respuesta en magnitud cae 3 dB[8][9], se definió wp=1.1 y  ws=0.7.

In [ ]:
# Visualización del pasa altos
w_rad = 2 * np.pi * np.linspace(0.1, fs/2, 1000) / fs
w, hh_hp = sig.freqz(num_hp, worN=w_rad)

plt.figure()
plt.plot(w * fs / (2 * np.pi), 20 * np.log10(np.abs(hh_hp) + 1e-15), label='Pasa altos')
plot_plantilla(filter_type='highpass', fpass=fpass_hp, ripple=1, fstop=fstop_hp, attenuation=attenuation, fs=fs)
plt.title('Filtro pasa altos', fontsize=18)
plt.xlabel('Frecuencia [Hz]', fontsize=16)
plt.ylabel('Magnitud [dB]', fontsize=16)
plt.tick_params(axis='both', labelsize=14)
plt.grid(which='both', axis='both')
plt.legend(fontsize=14)
plt.show()  

_Figura 3.1. Respuesta en frecuencia del filtro pasa altos - Respuesta en Magnitud_

Como puede observarse en la Figura 3.1, la respuesta en magnitud del filtro en la frecuencia de corte (1 Hz) cumple con ser de -3 dB.

Por otro lado, para el diseño de los filtros Butterworth de segundo orden únicamente la función únicamente exige seleccionar el orden (N=2) y las frecuencias de corte de las señales, que para cada filtro son: 2–4 Hz (delta), 4.5–7.5 Hz (theta), 8–12.5 Hz (alpha), 13–30 Hz (beta), 30–45 Hz (gamma).

In [1]:
# %% --- Diseño de Filtros IIR Butterworth ---

bandas_eeg = {
    'Delta': (2.0, 4.0),
    'Theta': (4.5, 7.5),
    'Alpha': (8.0, 12.5),
    'Beta':  (13.0, 30.0),
    'Gamma': (30.0, 45.0)
}

bandas = (['Delta','Theta','Alpha','Beta','Gamma'])

f_low_D, f_high_D = bandas_eeg['Delta']
f_low_T, f_high_T = bandas_eeg['Theta']
f_low_A, f_high_A = bandas_eeg['Alpha']
f_low_B, f_high_B = bandas_eeg['Beta']
f_low_G, f_high_G = bandas_eeg['Gamma']

# Diseño del filtro Butterworth de 2do orden
b_butter_D, a_butter_D = sig.butter(N=2, Wn=[f_low_D, f_high_D], btype='bandpass', fs=fs)
b_butter_T, a_butter_T = sig.butter(N=2, Wn=[f_low_T, f_high_T], btype='bandpass', fs=fs)
b_butter_A, a_butter_A = sig.butter(N=2, Wn=[f_low_A, f_high_A], btype='bandpass', fs=fs)
b_butter_B, a_butter_B = sig.butter(N=2, Wn=[f_low_B, f_high_B], btype='bandpass', fs=fs)
b_butter_G, a_butter_G = sig.butter(N=2, Wn=[f_low_G, f_high_G], btype='bandpass', fs=fs)

b_butter = [b_butter_D, b_butter_T, b_butter_A, b_butter_B, b_butter_G]
a_butter = [a_butter_D, a_butter_T, a_butter_A, a_butter_B, a_butter_G]

def plotear_funcion_transferencia_ab(b, a, fs, xlim = None, ylim = None, label = 'T(s)', polos_y_ceros = True, magnitud_fase_retardo = True, fig_inicial = 0, f_banda = None, fs_title=20, fs_label=18, fs_legend=14, fs_ticks=14):
    """
        Plotea la respuesta en frecuencia de una función transferencia de:
            - Módulo
            - Fase
            - Retardo de grupo
            - Polos y Ceros
        
        b: Vector de coeficientes que modifican la entrada del sistema
        a: Vector de coeficientes que modifican la salida del sistema
        xlim: Limites horizontales del gráfico
        xlim_s: Status del xlim (True si quiero limitar el gráfico, 
                                 False si quiero imprimirlo sin limites)
        fig_inicial: Número que quiero que tenga la primer figura a 
                    imprimir. Sirve para sobre-escribir (o no) plotteos 
                    de diferentes funciones en un mismo gráfico (dejar en 
                    0 si se quiere sobre-plotear)
        f_banda: Tupla con los límites de la banda a sombrear (ej: (8.0, 12.5))
        fs_title: Tamaño de la fuente para los títulos.
        fs_label: Tamaño de la fuente para las etiquetas de los ejes.
        fs_legend: Tamaño de la fuente para las leyendas.
        fs_ticks: Tamaño de la fuente para los números de los ejes.
    """
    
    # --- Módulo, Fase y Retardo de Fase ---
    f, h = sig.freqz(b, a, fs=fs) # worN = np.logspace(2,-2,1000) (dominio Z)
    phase = np.unwrap(np.angle(h)) # fase del grupo
    gd = -np.diff(phase) / (2 * np.pi * np.diff(f))
    
    # --- Ploteo ---
    figura = 1
    #if magnitud_fase_retardo == True:
        # Magnitud
    plt.figure(fig_inicial + figura)
    plt.title('Respuesta en Magnitud', fontsize=fs_title)
    plt.xlabel('Frecuencia [Hz]', fontsize=fs_label)
    plt.ylabel('|H(f)| [dB]', fontsize=fs_label)
    
    curva = plt.plot(f, 20*np.log10(abs(h)), label = f"Filtro {label}") 
    color_curva = curva[0].get_color()
    
    if f_banda is not None: 
        plt.axvspan(f_banda[0], f_banda[1], color=color_curva, alpha=0.15, label=f'Banda {label}', zorder=0)
        
        ticks_actuales = plt.gca().get_xticks() 
        nuevos_ticks = list(ticks_actuales) + [f_banda[0], f_banda[1]]
        nuevos_ticks = sorted(list(set(nuevos_ticks))) 
        
        if xlim is not None:
            nuevos_ticks = [t for t in nuevos_ticks if xlim[0] <= t <= xlim[1]]
            
        etiquetas_escalonadas = []
        nivel_abajo = False
        
        for i in range(len(nuevos_ticks)):
            if i > 0 and (nuevos_ticks[i] - nuevos_ticks[i-1]) < 2.0:
                nivel_abajo = not nivel_abajo 
            else:
                nivel_abajo = False
                
            texto = f"{nuevos_ticks[i]:g}" 
            
            if nivel_abajo:
                etiquetas_escalonadas.append(f"\n{texto}")
            else:
                etiquetas_escalonadas.append(texto)
                
        plt.xticks(nuevos_ticks, etiquetas_escalonadas, rotation=0)
            
        plt.tick_params(axis='both', labelsize=fs_ticks)
        
        if xlim is not None: plt.xlim(xlim)
        if ylim is not None: plt.ylim(ylim)
        plt.grid(True, which='both', ls=':')
        plt.legend(fontsize=fs_legend)
        figura+=1
    
        # Fase
        plt.figure(fig_inicial + figura)
        plt.title('Respuesta de Fase', fontsize=fs_title)
        plt.xlabel('Frecuencia [Hz]', fontsize=fs_label)
        plt.ylabel('Fase [°]', fontsize=fs_label)
        plt.tick_params(axis='both', labelsize=fs_ticks)
        
        plt.plot(f, np.degrees(phase), label = f"Filtro {label}")
        if xlim is not None: plt.xlim(xlim)
        plt.grid(True, which='both', ls=':')
        plt.legend(fontsize=fs_legend)
        figura+=1
        
        # Retardo de grupo
        plt.figure(fig_inicial + figura)
        plt.title('Retardo de Grupo', fontsize=fs_title)
        plt.xlabel('Frecuencia [Hz]', fontsize=fs_label)
        plt.ylabel('τg [s]', fontsize=fs_label)
        plt.tick_params(axis='both', labelsize=fs_ticks)
        
        plt.plot(f[:-1], gd, label = f"Filtro {label}")
        if xlim is not None: plt.xlim(xlim)
        plt.grid(True, which='both', ls=':')
        plt.legend(fontsize=fs_legend)
        figura+=1

# Compruebo respuesta en frecuencia de los filtros
for i in range(5):
    plotear_funcion_transferencia_ab(b = b_butter[i], a = a_butter[i], fs=fs, label = f'{bandas[i]}', xlim =[0, 55], ylim = [-60, 1], f_banda = bandas_eeg[bandas[i]])


NameError: name 'sig' is not defined

_Figura 3.2. Respuesta en frecuencia del filtro Butterworth - Respuesta en Magnitud. Las curvas representan la magnitud en [dB] con respecto a la frecuencia [Hz], mientras que el área de fondo representa la banda de paso deseada del filtro._

Como puede observarse en la Figura 3.2, los filtros ocupan correctamente su banda de paso con la banda de frecuencias deseada. A su vez, puede notarse el supuesto mencionado anteriormente: al ser filtros con caída suave, algunos podrían no estar atenuando correctamente frecuencias que no corresponden a la banda deseada. Por ejemplo: se puede observar de forma como el filtro que correspondiente a la banda ‘Beta’ (curva de color rojo) pasa por la banda ‘Alpha’ (banda de color verde) con una atenuación de entre -4 dB y -15 dB aproximadamente. Para determinar si la lenta caída de los filtros conlleva un solapamiento entre las bandas deseadas, se realizó un filtro Chebyshev tipo II.

Observando la figura 3.2 y teniendo en cuenta que luego se utilizará un filtrado bidireccional (el cual duplicará la atenuación y ripple del filtro), se definió diseñar los filtros Cheby2 con un ripple de 0.5 y una atenuación de 30 dB. A su vez, se determinó el ancho de la banda de transición para cada filtro de forma que decayeran 3 dB en sus frecuencias de corte.

In [ ]:
# %% --- Diseño de Filtros IIR chebyshev tipo II ---
ripple = 0.5 # dB
attenuation = 30 # dB
bdt = 0.7 # Ancho de la banda de transición
bdt_bp_izq = 0.2 # Distancia entre la frec. de corte y la frec. de paso
bdt_bp_der = 0.3

sos_cheby_D = sig.iirdesign(wp = [f_low_D + 0.1, f_high_D - 0.2], ws = [f_low_D - bdt, f_high_D + bdt], gpass = ripple, gstop = attenuation, 
                     analog = False, ftype = 'cheby2', output = 'sos', fs = fs)
sos_cheby_T = sig.iirdesign(wp = [f_low_T + bdt_bp_izq, f_high_T - bdt_bp_der], ws = [f_low_T - bdt, f_high_T + bdt], gpass = ripple, gstop = attenuation, 
                     analog = False, ftype = 'cheby2', output = 'sos', fs = fs)
sos_cheby_A = sig.iirdesign(wp = [f_low_A + bdt_bp_izq, f_high_A - bdt_bp_der], ws = [f_low_A - bdt, f_high_A + bdt], gpass = ripple, gstop = attenuation, 
                     analog = False, ftype = 'cheby2', output = 'sos', fs = fs)
sos_cheby_B = sig.iirdesign(wp = [f_low_B + bdt_bp_izq, f_high_B - bdt_bp_der], ws = [f_low_B - bdt, f_high_B + bdt], gpass = ripple, gstop = attenuation, 
                     analog = False, ftype = 'cheby2', output = 'sos', fs = fs)
sos_cheby_G = sig.iirdesign(wp = [f_low_G + bdt_bp_izq, f_high_G - bdt_bp_der], ws = [f_low_G - bdt, f_high_G + bdt], gpass = ripple, gstop = attenuation, 
                     analog = False, ftype = 'cheby2', output = 'sos', fs = fs)

sos_cheby = [sos_cheby_D, sos_cheby_T, sos_cheby_A, sos_cheby_B, sos_cheby_G]

def plotear_funcion_transferencia_sos(sos, fs, xlim = None, ylim = None, label = 'T(s)', polos_y_ceros = True, magnitud_fase_retardo = True, fig_inicial = 0, f_banda = None, fs_title=20, fs_label=18, fs_legend=14, fs_ticks=14):
    """
        Plotea la respuesta en frecuencia de una función transferencia de:
            - Módulo
            - Fase
            - Retardo de grupo
            - Polos y Ceros

        b: Vector de coeficientes que modifican la entrada del sistema
        a: Vector de coeficientes que modifican la salida del sistema
        xlim: Limites horizontales del gráfico
        xlim_s: Status del xlim (True si quiero limitar el gráfico, 
                                 False si quiero imprimirlo sin limites)
        fig_inicial: Número que quiero que tenga la primer figura a 
                    imprimir. Sirve para sobre-escribir (o no) plotteos 
                    de diferentes funciones en un mismo gráfico (dejar en 
                    0 si se quiere sobre-plotear)
        f_banda: Tupla con los límites de la banda a sombrear (ej: (8.0, 12.5))
        fs_title: Tamaño de la fuente para los títulos.
        fs_label: Tamaño de la fuente para las etiquetas de los ejes.
        fs_legend: Tamaño de la fuente para las leyendas.
        fs_ticks: Tamaño de la fuente para los números de los ejes.
    """

    # --- Módulo, Fase y Retardo de Fase ---
    f, h = sig.freqz_sos(sos, worN=8192, fs = fs)
    phase = np.unwrap(np.angle(h)) # fase del grupo
    gd = -np.diff(phase) / (2 * np.pi * np.diff(f))
    
    # --- Ploteo ---
    figura = 1
        # Magnitud
    plt.figure(fig_inicial + figura)
    plt.title('Respuesta en Magnitud', fontsize=fs_title)
    plt.xlabel('Frecuencia [Hz]', fontsize=fs_label)
    plt.ylabel('|H(f)| [dB]', fontsize=fs_label)

    curva = plt.plot(f, 20*np.log10(abs(h)), label = f"Filtro {label}") 
    color_curva = curva[0].get_color()
    
    if f_banda is not None: 
        plt.axvspan(f_banda[0], f_banda[1], color=color_curva, alpha=0.15, label=f'Banda {label}', zorder=0)
        
        ticks_actuales = plt.gca().get_xticks() 
        nuevos_ticks = list(ticks_actuales) + [f_banda[0], f_banda[1]]
        nuevos_ticks = sorted(list(set(nuevos_ticks))) 
        
        if xlim is not None:
            nuevos_ticks = [t for t in nuevos_ticks if xlim[0] <= t <= xlim[1]]
            
        etiquetas_escalonadas = []
        nivel_abajo = False
        
        for i in range(len(nuevos_ticks)):
            if i > 0 and (nuevos_ticks[i] - nuevos_ticks[i-1]) < 2.0:
                nivel_abajo = not nivel_abajo 
            else:
                nivel_abajo = False
                
            texto = f"{nuevos_ticks[i]:g}" 
            
            if nivel_abajo:
                etiquetas_escalonadas.append(f"\n{texto}")
            else:
                etiquetas_escalonadas.append(texto)
                
        plt.xticks(nuevos_ticks, etiquetas_escalonadas, rotation=0)
            
        plt.tick_params(axis='both', labelsize=fs_ticks)
        
        if xlim is not None: plt.xlim(xlim)
        if ylim is not None: plt.ylim(ylim)
        plt.grid(True, which='both', ls=':')
        plt.legend(fontsize=fs_legend)
        figura+=1
    
        # Fase
        plt.figure(fig_inicial + figura)
        plt.title('Respuesta de Fase', fontsize=fs_title)
        plt.xlabel('Frecuencia [Hz]', fontsize=fs_label)
        plt.ylabel('Fase [°]', fontsize=fs_label)
        plt.tick_params(axis='both', labelsize=fs_ticks)
        
        plt.plot(f, np.degrees(phase), label = f"Filtro {label}")
        if xlim is not None: plt.xlim(xlim)
        plt.grid(True, which='both', ls=':')
        plt.legend(fontsize=fs_legend)
        figura+=1
        
        # Retardo de grupo
        plt.figure(fig_inicial + figura)
        plt.title('Retardo de Grupo', fontsize=fs_title)
        plt.xlabel('Frecuencia [Hz]', fontsize=fs_label)
        plt.ylabel('τg [s]', fontsize=fs_label)
        plt.tick_params(axis='both', labelsize=fs_ticks)
        
        plt.plot(f[:-1], gd, label = f"Filtro {label}")
        if xlim is not None: plt.xlim(xlim)
        plt.grid(True, which='both', ls=':')
        plt.legend(fontsize=fs_legend)
        figura+=1

for i in range(5):
    plotear_funcion_transferencia_sos(sos = sos_cheby[i], label = f'{bandas[i]}', fs=fs, f_banda = bandas_eeg[bandas[i]], xlim = [0, 55], ylim = [-60, 1])


_Figura 3.3. Respuesta en frecuencia del filtro Chebyshev 2 - Respuesta en Magnitud_

Como puede observarse en la Figura 3.3, el filtro Chebyshev 2 presenta una caída en la banda de transición mucho más abrupta que la del filtro Butterworth visto en la Figura 3.2. Esta rápida caída se traduce en que el filtro genere ripple en su banda de ‘stop’, pero se obtiene la ventaja de que los filtros atenúen correctamente las bandas no deseadas a -30 dB, separando las bandas deseadas de forma más estricta que con el filtro utilizado por el estudio. 

Para comparar el resultado del filtrado mediante ambos filtros, se realizó un análisis espectral de potencias:


In [ ]:
# %% --- Análisis espectral en una banda - Comparativa entre filtros ---
bnd = 0 # Eligo la banda que quiero plotear

f_low, f_high = bandas_eeg[bandas[bnd]]

eeg_filtrado_banda_butter = sig.filtfilt(b = b_butter[bnd], a = a_butter[bnd], x = datos_canal_crudos)
eeg_filtrado_banda_cheby = sig.sosfiltfilt(sos=sos_cheby[bnd], x = datos_canal_crudos)

f_crudo, Px_crudo = welch(cant_promedio = 10, vector = datos_canal_crudos, zp = 2, fs=fs, window ='hann')
f_butter, Px_butter = welch(cant_promedio = 10, vector = eeg_filtrado_banda_butter, zp = 2, fs=fs, window ='hann')
f_cheby, Px_cheby = welch(cant_promedio = 10, vector = eeg_filtrado_banda_cheby, zp = 2, fs=fs, window ='hann')

plt.plot(f_crudo, 10*np.log10(Px_crudo + 1e-15), label='PSD crudo', color='lightgray')
plt.plot(f_butter, 10*np.log10(Px_butter + 1e-15), label=f'Filtrada con filtro {bandas[bnd]} Butterworth', linewidth=1.5)
plt.plot(f_cheby, 10*np.log10(Px_cheby + 1e-15), label=f'Filtrada con filtro {bandas[bnd]} Chebyshev II', linewidth=1.5)
plt.axvline(x=f_low, color='red', linestyle=':', linewidth=2)
plt.axvline(x=f_high, color='red', linestyle=':', linewidth=2)
plt.title(f'Análisis espectral de PSD: Comparativa entre filtros - Banda {bandas[bnd]}', fontsize = 20)
plt.xlabel('Frecuencia [Hz]', fontsize = 18)
plt.ylabel('PSD [dB]', fontsize = 18)
plt.legend(fontsize = 14, loc = 'upper right')
plt.grid(True)
plt.tick_params(axis='both', labelsize=14)
plt.xlim([f_low-1, f_high+1])
plt.ylim([-150,-100])
plt.show()

_Figura 3.4. Análisis Espectral: En el gráfico se comparan las estimaciones espectrales de la señal FP1 del sujeto H01 en la banda ‘Delta’ de frecuencias (2-4 Hz). La curva gris representa la estimación realizada a la señal sin filtrar. La azúl representa la señal filtrada mediante el filtro Butterworth y la naranja la filtrada mediante el filtro Chebyshev._

Como puede observarse en la Figura 3.4, el filtro Chebyshev logra atenuar mejor la potencia de la señal en los intervalos que están por fuera de la banda de paso deseada. A su vez, se puede ver que, debido a la suave caída del filtro Butterworth, hay intervalos de frecuencias como de 2 a 2.5 Hz o de 3.5 a 4 Hz donde el filtro no deja pasar toda la energía de la señal, atenuando una parte de la misma. El filtro Chebyshev, en cambio, presenta una caída mucho más pronunciada cerca de la frecuencia de corte, logrando conservar mejor la energía de señal.

A partir de las señales filtradas para los 28 sujetos en los 19 canales, se correlacionaron los datos uno a uno, obteniendo las siguientes matrices de conectividad (o correlación):

In [ ]:
# %% --- Matríz de conectividad para 1 sujeto y 1 banda - Butter ---
print("\nCalculando Matriz de Conectividad PLI filtrado con Butter...")

n_epocas = matriz_filtrada.shape[1] // muestras_por_epoca # Cuántas épocas enteras entran en el registro
datos_recortados = matriz_filtrada[:, :n_epocas * muestras_por_epoca]

# Remodelamos de (19, N) a (19, n_epocas, muestras_por_epoca) y luego transponemos a (n_epocas, 19, muestras_por_epoca)
datos_3d = datos_recortados.reshape(19, n_epocas, muestras_por_epoca).transpose(1, 0, 2)
print(f"Señal dividida en {n_epocas} épocas de {segundos_por_epoca} segundos.")

# Phase-Lag Index (PLI): Compara la diferencia de fase entre los canales para medir su correlación 
resultado_conectividad = spectral_connectivity_epochs(
    data=datos_3d,
    method='pli',
    sfreq=fs,
    fmin=fmin,
    fmax=fmax,
    faverage=True, # Promedia las frecuencias de la banda en un solo valor por conexión
    verbose=False
)

# El argumento 'dense' devuelve la matriz cuadrada completa
matriz_adyacencia = resultado_conectividad.get_data(output='dense')[:, :, 0]
matriz_adyacencia_butter = matriz_adyacencia + matriz_adyacencia.T # Le sumo la transpuesta para que me grafique toda la matriz

# %% --- Matríz de conectividad para 1 sujeto y 1 banda - Cheby ---
print("\nCalculando Matriz de Conectividad PLI filtrado con Cheby...")

matriz_filtrada = eeg_filtrado_sanos_cheby[sujeto][banda]

n_epocas = matriz_filtrada.shape[1] // muestras_por_epoca
datos_recortados = matriz_filtrada[:, :n_epocas * muestras_por_epoca]

# Remodelamos de (19, N) a (19, n_epocas, 500) y luego transponemos a (n_epocas, 19, 500)
datos_3d = datos_recortados.reshape(19, n_epocas, muestras_por_epoca).transpose(1, 0, 2)
print(f"Señal dividida en {n_epocas} épocas de {segundos_por_epoca} segundos.")

# Phase-Lag Index (PLI)
resultado_conectividad = spectral_connectivity_epochs(
    data=datos_3d,
    method='pli',
    sfreq=fs,
    fmin=fmin,
    fmax=fmax,
    faverage=True, # Promedia las frecuencias de la banda en un solo valor por conexión
    verbose=False
)

matriz_adyacencia = resultado_conectividad.get_data(output='dense')[:, :, 0]
matriz_adyacencia_cheby = matriz_adyacencia + matriz_adyacencia.T # Le sumo la transpuesta para que me grafique toda la matriz

# %% --- Visualización de matrices: Sujeto H01 con Butter vs Cheby en una unica banda de frecuencias ---
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Heatmap Sanos
sns.heatmap(
    matriz_adyacencia_butter, xticklabels=canales, yticklabels=canales, 
    cmap='jet', vmin=0, vmax=0.3, square=True, linewidths=0.5, ax=axes[0]
)
axes[0].set_title(f'Matriz de Conectividad - Sujeto H01 - PLI {banda} - Butter', fontsize = 16)
axes[0].set_xlabel('Canales', fontsize = 14)
axes[0].set_ylabel('Canales', fontsize = 14)

# Heatmap Esquizofrenia
sns.heatmap(
    matriz_adyacencia_cheby, xticklabels=canales, yticklabels=canales, 
    cmap='jet', vmin=0, vmax=0.3, square=True, linewidths=0.5, ax=axes[1]
)
axes[1].set_title(f'Matriz de Conectividad - Sujeto H01 - PLI {banda} - Cheby', fontsize = 16)
axes[1].set_xlabel('Canales', fontsize = 14)
axes[1].set_ylabel('Canales', fontsize = 14)

plt.tight_layout()
plt.show()


_Figura 3.5. Correlación de señales: Matriz de Correlación PLI del sujeto H01. Comparativa entre filtros Butterworth y Chebyshev_

Como puede observarse mediante las matrices, resulta complejo discernir una diferencia entre el uso de diferentes filtros, por lo que, al igual que en el estudio, se buscó representar las mismas como grafos (ver Figura 3.6 en el informe).

En la Figura 3.6 se puede observar la representación de las correlaciones entre señales en forma de red compleja o grafo, donde los resultados de los diferentes filtrados en la señal pueden ser comparados a partir de las características de las mismas. Como primer comparativa, se realizó el promedio de enlaces de cada red según si fueron filtradas mediante un filtro Butterworth o Chebyshev II, teniendo en cuenta también las dos poblaciones de sujetos:

<img src="./comparativa_cantidad_promedio.jpg" align="middle" width="400" /> 
_Figura 3.7. Análisis de redes: Cantidad de enlaces que obtuvieron en promedio las redes conformadas por señales filtradas mediante filtro Butterworth y Chebyshev II en la banda Delta de frecuencias._

Como puede observarse en la Figura 3.7, prácticamente no se encuentran diferencias entre utilizar un filtro Butterworth y un filtro Chebyshev II, por lo que se puede concluir que las características de la banda de transición del filtro y la minuciosa separación de las bandas de interés  no implican una diferencia relevante al momento de comparar la conectividad cerebral entre dos poblaciones.

# Discusión y Conclusiones
En este trabajo se comparó el uso de dos filtros con características en su banda de transición diferentes, recreando el preprocesamiento realizado en ‘Graph-based analysis of brain connectivity in schizophrenia’ por Elzbieta Olejarczyk y Wojciech Jernajczyk[1] para las diferentes señales de EEG disponibles. Se utilizaron herramientas de diseño de filtros IIR como ‘iir.design’ y ‘sig.butter’. A su vez, se justificó el uso de los mismos mediante el filtrado bidireccional y se resaltó el uso de filtros FIR para señales EEG que debieran ser filtradas al momento de ser adquiridas. Se analizó la respuesta en frecuencia de módulo de los filtros, comparando sus diseños para las distintas bandas de frecuencia. A su vez, se realizó un análisis espectral de potencias, estimando la energía de las señales mediante el método de Welch y comparando la energía de la señal antes y luego de ser filtrada mediante los diferentes filtros.

Adicionalmente, se realizó una comparativa a partir de la aplicación de los filtros utilizando correlaciones temporales, conformando matrices que luego serían representadas como grafos con el objetivo de realizar un análisis a partir de las características de las redes conformadas. Finalmente, se concluyó que la elección del filtro presenta diferencias en la energía de la señal de forma individual para una única señal, pero al momento de extrapolarlo a un análisis de varias señales, la diferencia entre los filtrados se vuelve mínima (o despreciable).


# Bilbiografía
[1] Olejarczyk E, Jernajczyk W (2017) Graph-based analysis of brain connectivity in schizophrenia. PLOS ONE 12(11): e0188629. https://doi.org/10.1371/journal.pone.0188629.

[2] Ramos-Argüelles, F., Morales, G., Egozcue, S., Pabón, R. M., & Alonso, M. T. (2009). Técnicas básicas de electroencefalografía: principios y aplicaciones clínicas. Anales del Sistema Sanitario de Navarra, 32, 69-82. https://scielo.isciii.es/scielo.php?script=sci_arttext&pid=S1137-66272009000600006&lng=es&nrm=iso&tlng=es

[3] Classification of natural sounds with EEG and Python - Neuro Questions - Neurostars. (s. f.). Recuperado 21 de mayo de 2026, de https://neurostars.org/t/classification-of-natural-sounds-with-eeg-and-python/19837

[4] Conceptos básicos sobre procesamiento digital de electroencefalograma y potenciales evocados - YouTube. (s. f.). Recuperado 21 de mayo de 2026, de https://www.youtube.com/watch?v=rfVL1JAQgjA

[5] Chiarion, G., Sparacino, L., Antonacci, Y., Faes, L., & Mesin, L. (2023). Connectivity Analysis in EEG Data: A Tutorial Review of the State of the Art and Emerging Trends. Bioengineering, 10(3), 372. https://doi.org/10.3390/BIOENGINEERING10030372

[6] mne.io.read_raw_edf — MNE 1.12.1 documentation. (s. f.). Recuperado 21 de mayo de 2026, de https://mne.tools/stable/generated/mne.io.read_raw_edf.html

[7] mne.preprocessing.ICA — MNE 1.12.1 documentation. (s. f.). Recuperado 29 de junio de 2026, de https://mne.tools/stable/generated/mne.preprocessing.ICA.html

[8] Holton, T. (s. f.). DIGITAL SIGNAL PROCESSING Principles and Applications. Recuperado www.cambridge.org/holton

[9] iirdesign — SciPy v1.18.0 Manual. (s. f.). Recuperado 7 de julio de 2026, de https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.iirdesign.html

[10] Artifacts. (s. f.). Recuperado 13 de julio de 2026, de https://www.learningeeg.com/artifacts
